In [ ]:
import time
from collections.abc import Callable

import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import optax
import optimistix as optx


jax.config.update("jax_enable_x64", True)

# Helmholtz parameters
A1 = 1.0
A2 = 4.0
K = 1.0

# Domain
X0 = -1.0
Y0 = -1.0
LX = 2.0
LY = 2.0

# Network
UNITS = 30
KMAX = 1
SEED = 1234

N_F = 30000
N_CHANGE = 200
RAD_K1 = 1.0
RAD_K2 = 0.0
RAD_ARGS = (RAD_K1, RAD_K2)

ADAM_EPOCHS = 0         
ADAM_PRINT_EVERY = 100
ADAM_LR = 1e-4

QN_STEPS = 50000
QN_METHOD = "broyden_trust"   
RTOL = 1e-16
ATOL = 1e-16



def generate_inputs(key: jr.PRNGKey, n_points: int) -> jnp.ndarray:
    xy_unit = jr.uniform(key, shape=(n_points, 2))
    x = X0 + LX * xy_unit[:, 0:1]
    y = Y0 + LY * xy_unit[:, 1:2]
    return jnp.hstack([x, y])


def adaptive_rad(network, key: jr.PRNGKey, n_points: int, rad_args, n_test: int = 50000):
    k1, k2 = rad_args
    key, key_test, key_choice = jr.split(key, 3)

    x_test = generate_inputs(key_test, n_test)
    residual = jax.vmap(helmholtz_residual, in_axes=(None, 0, 0))(
        network,
        x_test[:, 0],
        x_test[:, 1],)

    eps = 1e-12
    residual_abs = jnp.abs(residual).reshape(-1)
    weights = jnp.power(residual_abs + eps, float(k1))
    weights = weights / jnp.maximum(jnp.mean(weights), eps) + float(k2)

    probs = weights / jnp.maximum(jnp.sum(weights), eps)
    probs = jnp.clip(probs, a_min=0.0, a_max=jnp.inf)
    probs = probs / jnp.maximum(jnp.sum(probs), eps)

    n_choose = min(n_points, x_test.shape[0])
    idx = jr.choice(
        key_choice,
        x_test.shape[0],
        shape=(n_choose,),
        replace=False,
        p=probs,)
    return x_test[idx], key



class PeriodicC0Layer(eqx.Module):
    kmax: int
    x0: float
    y0: float
    lx: float
    ly: float

    def __init__(self, kmax: int = 1, x0: float = -1.0, y0: float = -1.0, lx: float = 2.0, ly: float = 2.0):
        self.kmax = kmax
        self.x0 = x0
        self.y0 = y0
        self.lx = lx
        self.ly = ly

    def __call__(self, inputs: jnp.ndarray) -> jnp.ndarray:
        x = inputs[..., 0:1]
        y = inputs[..., 1:2]

        x_shift = x - self.x0
        y_shift = y - self.y0

        ks = jnp.arange(1, self.kmax + 1, dtype=inputs.dtype)

        x_periodic = 2.0 * jnp.pi * x_shift * ks / self.lx
        y_periodic = 2.0 * jnp.pi * y_shift * ks / self.ly

        return jnp.concatenate(
            [
                jnp.cos(x_periodic),
                jnp.sin(x_periodic),
                jnp.cos(y_periodic),
                jnp.sin(y_periodic),],
            axis=-1,)


class Helmholtz2D(eqx.Module):
    periodic_layer: PeriodicC0Layer
    layers: list

    def __init__(self, key, units: int = 30, kmax: int = 1, x0: float = -1.0, y0: float = -1.0, lx: float = 2.0, ly: float = 2.0):
        self.periodic_layer = PeriodicC0Layer(kmax=kmax, x0=x0, y0=y0, lx=lx, ly=ly)

        keys = jr.split(key, 5)
        in_dim = 4 * kmax

        self.layers = [
            eqx.nn.Linear(in_dim, units, key=keys[0]),
            eqx.nn.Linear(units, units, key=keys[1]),
            eqx.nn.Linear(units, units, key=keys[2]),
            eqx.nn.Linear(units, units, key=keys[3]),
            eqx.nn.Linear(units, 1, key=keys[4]),]

    def __call__(self, x, y):
        xy = jnp.stack([x, y], axis=-1)
        z = self.periodic_layer(xy)

        for layer in self.layers[:-1]:
            z = jax.nn.tanh(layer(z))
        return self.layers[-1](z)


initializer = jax.nn.initializers.glorot_normal()


def init_weight(weight: jax.Array, key: jr.PRNGKey) -> jax.Array:
    out_dim, in_dim = weight.shape
    return initializer(key, shape=(out_dim, in_dim))


def init_linear_layers(model, init_fn, key):
    is_linear = lambda layer: isinstance(layer, eqx.nn.Linear)

    get_weights = lambda m: [
        layer.weight
        for layer in jax.tree_util.tree_leaves(m, is_leaf=is_linear)
        if is_linear(layer)]
    
    get_biases = lambda m: [
        layer.bias
        for layer in jax.tree_util.tree_leaves(m, is_leaf=is_linear)
        if is_linear(layer)]

    weights = get_weights(model)
    biases = get_biases(model)

    new_weights = [
        init_fn(weight, subkey)
        for weight, subkey in zip(weights, jr.split(key, len(weights)))]
    new_biases = jax.tree_util.tree_map(jnp.zeros_like, biases)

    model = eqx.tree_at(get_weights, model, new_weights)
    model = eqx.tree_at(get_biases, model, new_biases)
    return model



def source_term(x, y):
    exact = jnp.sin(A1 * jnp.pi * x) * jnp.sin(A2 * jnp.pi * y)
    return (K**2 - (A1 * jnp.pi) ** 2 - (A2 * jnp.pi) ** 2) * exact


@eqx.filter_jit
def helmholtz_residual(network, x, y):
    u_fn = lambda x_, y_: network(x_, y_)[0]
    u_x = jax.grad(u_fn, argnums=0)
    u_y = jax.grad(u_fn, argnums=1)
    u_xx = jax.grad(u_x, argnums=0)(x, y)
    u_yy = jax.grad(u_y, argnums=1)(x, y)
    u = u_fn(x, y)
    return u_xx + u_yy + (K**2) * u - source_term(x, y)


@eqx.filter_jit
def loss_fn(network, xy_r):
    residual = jax.vmap(helmholtz_residual, in_axes=(None, 0, 0))( network, xy_r[:, 0], xy_r[:, 1],)
    return jnp.mean(jnp.square(residual))



def run_adam_warmup(network, xy_f, key):
    if ADAM_EPOCHS <= 0:
        print("[ADAM] skipped")
        return network, xy_f, key

    optimizer = optax.radam(learning_rate=ADAM_LR)
    opt_state = optimizer.init(eqx.filter(network, eqx.is_inexact_array))

    @eqx.filter_jit
    def train_step(model, state, xy_r):
        loss_value, grads = eqx.filter_value_and_grad(loss_fn)(model, xy_r)
        updates, new_state = optimizer.update(grads, state, model)
        new_model = eqx.apply_updates(model, updates)
        return new_model, new_state, loss_value

    key, key_resample = jr.split(key)
    t0 = time.time()

    for epoch in range(ADAM_EPOCHS):
        if (epoch + 1) % N_CHANGE == 0:
            xy_f, key_resample = adaptive_rad(network, key_resample, N_F, RAD_ARGS)

        network, opt_state, loss_value = train_step(network, opt_state, xy_f)

        if epoch == 0 or (epoch + 1) % ADAM_PRINT_EVERY == 0:
            print(f"[ADAM] epoch={epoch + 1} loss={float(loss_value):.7e}")

    print(f"[ADAM] elapsed={time.time() - t0:.2f}s")
    return network, xy_f, key



class BFGSTrustRegion(optx.AbstractBFGS):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.LinearTrustRegion()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class SSBFGSTrustRegion(optx.AbstractSSBFGS):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.LinearTrustRegion()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class SSBFGSBacktracking(optx.AbstractSSBFGS):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = False
    search: optx.AbstractSearch = optx.BacktrackingArmijo()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class BroydenTrustRegion(optx.AbstractSSBroyden):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.LinearTrustRegion()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class BroydenBacktrackingStrongWolfe(optx.AbstractSSBroyden):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.BacktrackingStrongWolfe()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


def make_qn_solver(method_name: str):
    verbose = frozenset({"loss"})

    if method_name == "bfgs_trust":
        solver = BFGSTrustRegion(rtol=RTOL, atol=ATOL, verbose=verbose)
    elif method_name == "ssbfgs_trust":
        solver = SSBFGSTrustRegion(rtol=RTOL, atol=ATOL, verbose=verbose)
    elif method_name == "ssbfgs_backtracking":
        solver = SSBFGSBacktracking(rtol=RTOL, atol=ATOL, verbose=verbose)
    elif method_name == "broyden_trust":
        solver = BroydenTrustRegion(rtol=RTOL, atol=ATOL, verbose=verbose)
    elif method_name == "broyden_strong_wolfe":
        solver = BroydenBacktrackingStrongWolfe(rtol=RTOL, atol=ATOL, verbose=verbose)
    else:
        raise ValueError(f"Unknown QN_METHOD: {method_name}")

    return optx.BestSoFarMinimiser(solver)


def run_quasi_newton(network, xy_f):
    params, static = eqx.partition(network, eqx.is_inexact_array)

    def objective(dynamic_model, static_model):
        model = eqx.combine(dynamic_model, static_model)
        return loss_fn(model, xy_f)

    solver = make_qn_solver(QN_METHOD)

    print(f"[{QN_METHOD}] training started")
    t0 = time.time()

    solution = optx.minimise(
        objective,
        solver,
        params,
        args=static,
        max_steps=QN_STEPS,
        throw=False,)

    trained_network = eqx.combine(solution.value, static)
    final_loss = float(loss_fn(trained_network, xy_f))

    print(f"[{QN_METHOD}] elapsed={time.time() - t0:.2f}s")
    print(f"[{QN_METHOD}] steps={solution.stats['num_steps']}")
    print(f"[{QN_METHOD}] final loss={final_loss:.7e}")

    return trained_network, solution



def main():
    print(f"PINN for 2D Helmholtz: A1={A1}, A2={A2}, K={K}")
    print(f"Quasi-Newton method: {QN_METHOD}")

    key = jr.PRNGKey(SEED)

    key, data_key = jr.split(key)
    xy_f = generate_inputs(data_key, N_F)

    key, init_key = jr.split(key)
    pinn = Helmholtz2D(init_key, units=UNITS, kmax=KMAX, x0=X0, y0=Y0, lx=LX, ly=LY)
    pinn = init_linear_layers(pinn, init_weight, init_key)

    pinn, xy_f, key = run_adam_warmup(pinn, xy_f, key)
    pinn, qn_solution = run_quasi_newton(pinn, xy_f)

    return pinn, qn_solution


if __name__ == "__main__":
    pinn, qn_solution = main()
